# SatyaScore — GPU runbook (Colab T4)

Runs the model-bound work (latency benchmark, retrieval diagnostic, and — behind a flag —
signal recompute + result regeneration) on a GPU and pushes generated results back to the repo.

**Every cell is re-runnable from scratch** (Colab sessions drop without warning). Only manual steps:
paste the GitHub token in Cell 2, and flip the two flags in Cells 8–9 if you want the heavy steps.

## Cell 1 — GPU guard
Stop immediately if no GPU was allocated; every downstream number is meaningless on CPU.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), (
    "No GPU allocated. Runtime -> Change runtime type -> T4 GPU, then re-run this cell."
)
print("CUDA OK:", torch.cuda.get_device_name(0))

## Cell 2 — Clone or update the (private) repo
Token read via getpass — never hardcoded, never printed. It is scrubbed from the stored remote so it can't land in `.git/config`.

In [ ]:
import os, getpass
from pathlib import Path

GH_TOKEN = getpass.getpass("GitHub token (input hidden): ")  # kept in memory only
_url = "https://" + GH_TOKEN + "@github.com/ViveekB410/SatyaScore.git"
if Path("/content/SatyaScore/.git").exists():
    os.chdir("/content/SatyaScore")
    os.system("git pull " + _url + " main")          # os.system: token not echoed to output
else:
    os.system("git clone " + _url + " /content/SatyaScore")
    os.chdir("/content/SatyaScore")
os.system("git remote set-url origin https://github.com/ViveekB410/SatyaScore.git")  # scrub token
print("repo ready at", os.getcwd())

## Cell 3 — Install
Colab's CUDA torch is already correct (`requirements.txt` only asks `>=2.0.0`), so this won't reinstall it. `WANDB_MODE=disabled` stops wandb prompting.

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q
print("install done")

## Cell 4 — Confirm the device
Both must print `cuda`. This is the checkpoint that catches a silent CPU fallback before any hours are spent.

In [ ]:
from member1_model.nli_scorer import NLIScorer
from member1_model.embeddings import CrossLingualEmbedder
print("NLI device     :", NLIScorer().device)
print("Embedder device:", CrossLingualEmbedder().device)

## Cell 5 — Test sanity
Expect ~110 passing (108 baseline + the batching + span tests). If tests fail here, the environment is wrong — don't trust later numbers.

In [ ]:
!python -m pytest -q

## Cell 6 — Task A: latency benchmark
Reads the per-stage `timings_ms`; states whether the PRD targets are met on this GPU. Writes `results/latency_benchmark.md`.

In [ ]:
import os
from IPython.display import Markdown, display
if os.path.exists("benchmark_latency.py"):
    !python benchmark_latency.py
    if os.path.exists("results/latency_benchmark.md"):
        display(Markdown(open("results/latency_benchmark.md", encoding="utf-8").read()))
else:
    print("benchmark_latency.py not found — see PHASE3_IMPROVEMENTS.md Task A")

## Cell 7 — Task B: retrieval diagnostic
**This decides the direction of the fixes.** It states explicitly whether the evidence points to a retrieval (C1), granularity (C2), or model/label (C3) problem. Writes `results/retrieval_diagnostic.{md,csv}`.

In [ ]:
import os
from IPython.display import Markdown, display
if os.path.exists("diagnose_retrieval.py"):
    !python diagnose_retrieval.py
    if os.path.exists("results/retrieval_diagnostic.md"):
        display(Markdown(open("results/retrieval_diagnostic.md", encoding="utf-8").read()))
else:
    print("diagnose_retrieval.py not found — see PHASE3_IMPROVEMENTS.md Task B")

## Cell 8 — Task D: recompute + regenerate (GUARDED)
Overwrites generated results. Flip `RUN_FULL_RECOMPUTE = True` **deliberately**, and only after reading Cell 7 and doing a Task C fix. Regenerates as a set (PHASE3 rule 3).

In [ ]:
RUN_FULL_RECOMPUTE = False  # <-- flip to True on purpose

if RUN_FULL_RECOMPUTE:
    !python recompute_signals.py --limit 6      # smoke test first
    !python recompute_signals.py                # full (writes gold_final_scored.csv)
    !python make_clean_split.py                 # -> gold_train_clean.csv / gold_test_clean.csv
    !python -m member1_model.train_weights --csv gold_train_clean.csv --languages en hi
    !python honest_ablation.py --train gold_train_clean.csv --test gold_test_clean.csv --out results/ablation_clean_no_semsim.json --table ablation_clean_no_semsim.md
    !python generate_results_summary.py         # -> RESULTS_SUMMARY.md
    print("recompute + regeneration complete")
else:
    print("RUN_FULL_RECOMPUTE is False — skipping (flip to True to regenerate everything).")

## Cell 9 — Optional: rebuild the bigger demo corpus (GUARDED)
GPU makes a larger demo corpus practical. Affects the live API demo only, never the evaluation.

In [ ]:
REBUILD_DEMO_CORPUS = False  # <-- flip to True on purpose

if REBUILD_DEMO_CORPUS:
    !python build_demo_corpus.py
    print("demo corpus rebuilt")
else:
    print("REBUILD_DEMO_CORPUS is False — skipping.")

## Cell 10 — Commit + push generated results
Stages only generated outputs (never `.venv`, caches, or checkpoints). Uses the in-memory token for the push URL (not printed, not committed).

In [ ]:
!git config user.email "colab@satyascore.local"
!git config user.name  "SatyaScore Colab"
!git add results Figures RESULTS_SUMMARY.md HANDOFF.md ablation_clean_no_semsim.md 2>/dev/null
!git status --short
import subprocess, os
r = subprocess.run(["git", "commit", "-m", "Colab GPU: regenerated results (benchmark / diagnostic / eval)"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
os.system("git push https://" + GH_TOKEN + "@github.com/ViveekB410/SatyaScore.git HEAD:main")
print("push attempted (no-op if nothing was committed)")